# Naive Bayes Baseline Comparison

这个 notebook 用于在 `conda` 环境中复现实验：把 `Naive Bayes` 作为一个 **traditional probabilistic / weak retrieval baseline**，与当前的 `语义召回 + 标签召回 + Cross-Encoder 精排` 做对照。

运行前提：

1. 先执行 `conda activate 7606`
2. 从该 conda 环境启动 Jupyter，并用该环境对应的 kernel 打开本 notebook
   - 如果该环境里还没有 notebook 运行组件，先安装 `jupyter` / `notebook` / `ipykernel`
3. 保证 `app/.env` 可用
4. 保证以下数据文件存在：
   - `match_data_preprocessing/data/enhanced_drug_table_v1_structured.csv`
   - `data/eval_dataset_llm.json`
   - `drug_comprehensive_embeddings.npy`

实验分两层：

- 召回层公平对比
- 受控端到端对比

In [ ]:
import os
import platform
import sys

import sklearn
import torch
import transformers

EXPECTED_CONDA_ENV = "7606"
python_executable = sys.executable.replace('\\', '/')
conda_default_env = os.environ.get('CONDA_DEFAULT_ENV', '')
path_match = f"/envs/{EXPECTED_CONDA_ENV}/" in python_executable
name_match = conda_default_env == EXPECTED_CONDA_ENV
is_expected_env = path_match or name_match

print('sys.executable =', sys.executable)
print('python version =', platform.python_version())
print('CONDA_DEFAULT_ENV =', conda_default_env or '<empty>')
print('scikit-learn =', sklearn.__version__)
print('transformers =', transformers.__version__)
print('torch =', torch.__version__)

assert is_expected_env, (
    'Kernel is not running from the expected conda env `7606`. '\
    'Please run `conda activate 7606` and start Jupyter from that env.'
)


> Evaluation scope note:
>
> 这个 notebook 比较的是 **drug retrieval / ranking performance**，不是 clinical prescription correctness。
>
> `Naive Bayes` 在这里被当作 **weak retrieval baseline**，用来回答：一个轻量、传统的概率模型，与当前双召回主线相比差距有多大。

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from tqdm.auto import tqdm


def find_repo_root(start: Path) -> Path:
    search_roots = [start.resolve(), *start.resolve().parents]
    for candidate in search_roots:
        if (candidate / 'requirements_mac.txt').exists() and (candidate / 'app').exists():
            return candidate
        nested = candidate / 'ARIN7102_Group_Project'
        if (nested / 'requirements_mac.txt').exists() and (nested / 'app').exists():
            return nested
    raise FileNotFoundError('Could not locate ARIN7102_Group_Project repo root from current working directory.')


CWD = Path.cwd().resolve()
REPO_ROOT = find_repo_root(CWD)
APP_DIR = REPO_ROOT / 'app'
INTERACTIONS_DIR = APP_DIR / 'interactions'
DATA_DIR = REPO_ROOT / 'data'
MATCH_DATA_DIR = REPO_ROOT / 'match_data_preprocessing' / 'data'
DRUG_TABLE_PATH = MATCH_DATA_DIR / 'enhanced_drug_table_v1_structured.csv'
EVAL_DATASET_PATH = DATA_DIR / 'eval_dataset_llm.json'
EMBEDDING_PATH = REPO_ROOT / 'drug_comprehensive_embeddings.npy'
ENV_PATH = APP_DIR / '.env'

assert DRUG_TABLE_PATH.exists(), f'Drug table not found: {DRUG_TABLE_PATH}'
assert EVAL_DATASET_PATH.exists(), f'Eval dataset not found: {EVAL_DATASET_PATH}'
assert EMBEDDING_PATH.exists(), f'Embedding file not found: {EMBEDDING_PATH}'
assert ENV_PATH.exists(), f'.env file not found: {ENV_PATH}'

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Keep cwd aligned with existing project docs and .env discovery expectations.
os.chdir(APP_DIR)

from app.embedded_module import build_query_text, prepare_drug_dataframe
from app.evaluation.metrics import evaluate_batch, evaluate_single_query
from app.fastapi_module.service import get_recommendation_service

print('REPO_ROOT =', REPO_ROOT)
print('APP_DIR =', APP_DIR)
print('DRUG_TABLE_PATH =', DRUG_TABLE_PATH)
print('EVAL_DATASET_PATH =', EVAL_DATASET_PATH)
print('EMBEDDING_PATH =', EMBEDDING_PATH)
print('ENV_PATH =', ENV_PATH, '| size =', ENV_PATH.stat().st_size, 'bytes')
if ENV_PATH.stat().st_size == 0:
    print('WARNING: app/.env exists but is empty. Some modules that depend on secrets may fail outside this notebook.')


In [ ]:
CANDIDATE_TOP_K = 300
FINAL_TOP_K = 20
K_VALUES = [5, 10, 20]
EVAL_LIMIT = None  # Set an int here for a quick smoke test.

df_raw = pd.read_csv(DRUG_TABLE_PATH)
df = prepare_drug_dataframe(df_raw)

with open(EVAL_DATASET_PATH, encoding='utf-8') as f:
    eval_samples = json.load(f)

if EVAL_LIMIT is not None:
    eval_samples = eval_samples[:EVAL_LIMIT]

assert isinstance(eval_samples, list) and len(eval_samples) > 0, 'Evaluation dataset is empty or malformed.'

print('drug rows =', len(df))
print('eval samples =', len(eval_samples))
print('candidate_top_k =', CANDIDATE_TOP_K)
print('final_top_k =', FINAL_TOP_K)

display(df[['drug_name', 'matched_disease_keys', 'matched_symptoms', 'avg_rating']].head(3))
display(pd.DataFrame(eval_samples[:2]))


In [ ]:
X_train_text = df['semantic_text'].fillna('[MASK]').astype(str)
y_train = df.index.to_numpy()

vectorizer = CountVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=20000,
)
nb_model = MultinomialNB(alpha=1.0, fit_prior=False)

nb_fit_start = time.perf_counter()
X_train = vectorizer.fit_transform(X_train_text)
nb_model.fit(X_train, y_train)
nb_fit_seconds = time.perf_counter() - nb_fit_start

print('NB vectorized matrix shape =', X_train.shape)
print('NB classes =', len(nb_model.classes_))
print('NB fit time (seconds) =', round(nb_fit_seconds, 3))


In [ ]:
service_init_start = time.perf_counter()
service = get_recommendation_service()
service.ensure_ready()
pipeline = service.pipeline
service_init_seconds = time.perf_counter() - service_init_start

print('Current pipeline is ready.')
print('Pipeline init time (seconds) =', round(service_init_seconds, 3))


In [ ]:
def min_max_normalize(values) -> np.ndarray:
    arr = np.asarray(values, dtype=np.float32)
    if arr.size == 0:
        return arr
    min_value = float(arr.min())
    max_value = float(arr.max())
    if max_value - min_value < 1e-12:
        return np.ones_like(arr, dtype=np.float32)
    return ((arr - min_value) / (max_value - min_value)).astype(np.float32)


def extract_label_names(items) -> list[str]:
    names = []
    for item in items or []:
        if isinstance(item, dict):
            if 'name' in item:
                name = str(item.get('name', '')).strip()
            elif len(item) == 1:
                name = str(next(iter(item))).strip()
            else:
                name = ''
            if name:
                names.append(name)
    return names


def build_nb_query(sample: dict, include_labels: bool) -> str:
    symptom_text = str(sample.get('symptom_text', '') or '').strip()
    if not include_labels:
        return symptom_text or '[MASK]'

    disease_names = extract_label_names(sample.get('diseases', []))
    symptom_names = extract_label_names(sample.get('symptoms', []))
    label_text = build_query_text(disease_names, symptom_names)
    parts = [part for part in [symptom_text, label_text] if part and part != '[MASK]']
    return ' '.join(parts) if parts else '[MASK]'


def retrieve_nb_candidates(sample: dict, include_labels: bool, candidate_top_k: int = CANDIDATE_TOP_K) -> pd.DataFrame:
    query_text = build_nb_query(sample, include_labels=include_labels)
    query_matrix = vectorizer.transform([query_text])
    log_proba = nb_model.predict_log_proba(query_matrix)[0]
    raw_scores = np.exp(log_proba).astype(np.float32)
    top_positions = np.argsort(raw_scores)[::-1][:candidate_top_k]
    top_indices = nb_model.classes_[top_positions]

    candidates = df.loc[top_indices].copy()
    candidates['nb_score_raw'] = raw_scores[top_positions]
    candidates = candidates.sort_values('nb_score_raw', ascending=False).copy()
    candidates['recall_fused_score'] = min_max_normalize(candidates['nb_score_raw'].to_numpy())
    return candidates


def recommendation_record(sample: dict, ranked_df: pd.DataFrame) -> dict:
    return {
        'query_id': sample['query_id'],
        'recommended': ranked_df['drug_name'].astype(str).tolist(),
        'relevant': sample['relevant_drugs'],
        'relevance_scores': sample.get('relevance_scores', {}),
    }


def collect_top_rankings(query_id: str, method: str, ranked_df: pd.DataFrame, score_column: str | None, top_n: int = FINAL_TOP_K) -> list[dict]:
    subset = ranked_df.copy()
    if score_column and score_column in subset.columns:
        subset = subset.sort_values(score_column, ascending=False)
    rows = []
    for rank, row in enumerate(subset.head(top_n).itertuples(index=False), start=1):
        rows.append({
            'query_id': query_id,
            'method': method,
            'rank': rank,
            'drug_name': getattr(row, 'drug_name', ''),
            'score': float(getattr(row, score_column)) if score_column and hasattr(row, score_column) else np.nan,
        })
    return rows


def run_recall_layer_benchmark(samples: list[dict]):
    method_records = {
        'semantic recall only': [],
        'label recall only': [],
        'dual recall without rerank': [],
        'NB-text': [],
        'NB-text+labels': [],
    }
    per_query_rows = []
    timing_rows = []
    ranking_rows = []
    score_column_map = {
        'semantic recall only': 'semantic_score',
        'label recall only': 'label_score',
        'dual recall without rerank': 'recall_fused_score',
        'NB-text': 'nb_score_raw',
        'NB-text+labels': 'nb_score_raw',
    }

    for sample in tqdm(samples, desc='Recall-layer comparison'):
        query_id = sample['query_id']

        start = time.perf_counter()
        semantic_df = pipeline.semantic_recall(sample['symptom_text'], top_k=CANDIDATE_TOP_K)
        timing_rows.append({'query_id': query_id, 'method': 'semantic recall only', 'latency_seconds': time.perf_counter() - start})

        start = time.perf_counter()
        label_df = pipeline.label_recall(sample['diseases'], sample['symptoms'], top_k=CANDIDATE_TOP_K)
        timing_rows.append({'query_id': query_id, 'method': 'label recall only', 'latency_seconds': time.perf_counter() - start})

        start = time.perf_counter()
        fused_df = pipeline.fuse_recalls(
            semantic_candidates=semantic_df,
            label_candidates=label_df,
            semantic_weight=0.5,
            label_weight=0.5,
            top_k=CANDIDATE_TOP_K,
        )
        timing_rows.append({'query_id': query_id, 'method': 'dual recall without rerank', 'latency_seconds': time.perf_counter() - start})

        start = time.perf_counter()
        nb_text_df = retrieve_nb_candidates(sample, include_labels=False, candidate_top_k=CANDIDATE_TOP_K)
        timing_rows.append({'query_id': query_id, 'method': 'NB-text', 'latency_seconds': time.perf_counter() - start})

        start = time.perf_counter()
        nb_text_labels_df = retrieve_nb_candidates(sample, include_labels=True, candidate_top_k=CANDIDATE_TOP_K)
        timing_rows.append({'query_id': query_id, 'method': 'NB-text+labels', 'latency_seconds': time.perf_counter() - start})

        ranked_by_method = {
            'semantic recall only': semantic_df,
            'label recall only': label_df,
            'dual recall without rerank': fused_df,
            'NB-text': nb_text_df,
            'NB-text+labels': nb_text_labels_df,
        }

        for method, ranked_df in ranked_by_method.items():
            record = recommendation_record(sample, ranked_df)
            method_records[method].append(record)
            metrics = evaluate_single_query(
                recommended=record['recommended'],
                relevant=record['relevant'],
                relevance_scores=record['relevance_scores'],
                k_values=K_VALUES,
            )
            per_query_rows.append({'query_id': query_id, 'method': method, **metrics})
            ranking_rows.extend(collect_top_rankings(query_id, method, ranked_df, score_column_map[method]))

    summary_df = pd.DataFrame([
        {'method': method, **evaluate_batch(records, k_values=K_VALUES)}
        for method, records in method_records.items()
    ])
    return method_records, pd.DataFrame(per_query_rows), summary_df, pd.DataFrame(timing_rows), pd.DataFrame(ranking_rows)


def run_end_to_end_benchmark(samples: list[dict]):
    method_records = {
        'current_full_pipeline': [],
        'nb_text_labels_same_rerank': [],
    }
    per_query_rows = []
    timing_rows = []
    ranking_rows = []

    for sample in tqdm(samples, desc='End-to-end comparison'):
        query_id = sample['query_id']

        start = time.perf_counter()
        current_df = service.recommend(
            symptom_text=sample['symptom_text'],
            diseases=sample['diseases'],
            symptoms=sample['symptoms'],
            top_k=FINAL_TOP_K,
            recall_top_k_each=CANDIDATE_TOP_K,
            fused_top_k=CANDIDATE_TOP_K,
            recall_weight_semantic=0.5,
            recall_weight_label=0.5,
        )
        timing_rows.append({'query_id': query_id, 'method': 'current_full_pipeline', 'latency_seconds': time.perf_counter() - start})

        start = time.perf_counter()
        nb_candidates = retrieve_nb_candidates(sample, include_labels=True, candidate_top_k=CANDIDATE_TOP_K)
        nb_reranked_df = pipeline.rerank(
            fused_candidates=nb_candidates,
            symptom_text=sample['symptom_text'],
            disease_items=sample['diseases'],
            symptom_items=sample['symptoms'],
            top_k=FINAL_TOP_K,
            final_weight_recall=0.35,
            final_weight_cross_encoder=0.50,
            final_weight_business=0.15,
        )
        timing_rows.append({'query_id': query_id, 'method': 'nb_text_labels_same_rerank', 'latency_seconds': time.perf_counter() - start})

        ranked_by_method = {
            'current_full_pipeline': current_df,
            'nb_text_labels_same_rerank': nb_reranked_df,
        }

        for method, ranked_df in ranked_by_method.items():
            record = recommendation_record(sample, ranked_df)
            method_records[method].append(record)
            metrics = evaluate_single_query(
                recommended=record['recommended'],
                relevant=record['relevant'],
                relevance_scores=record['relevance_scores'],
                k_values=K_VALUES,
            )
            per_query_rows.append({'query_id': query_id, 'method': method, **metrics})
            ranking_rows.extend(collect_top_rankings(query_id, method, ranked_df, 'final_score'))

    summary_df = pd.DataFrame([
        {'method': method, **evaluate_batch(records, k_values=K_VALUES)}
        for method, records in method_records.items()
    ])
    return method_records, pd.DataFrame(per_query_rows), summary_df, pd.DataFrame(timing_rows), pd.DataFrame(ranking_rows)


def build_query_metadata(samples: list[dict]) -> pd.DataFrame:
    return pd.DataFrame([
        {
            'query_id': sample['query_id'],
            'symptom_text': sample['symptom_text'],
            'relevant_drugs': ', '.join(sample.get('relevant_drugs', [])),
        }
        for sample in samples
    ])


def top_drugs(rankings_df: pd.DataFrame, query_id: str, method: str, top_n: int = 5) -> pd.DataFrame:
    return rankings_df[(rankings_df['query_id'] == query_id) & (rankings_df['method'] == method)].head(top_n).reset_index(drop=True)


## Recall-Layer Comparison

这一节只比较候选召回质量，不引入 Cross-Encoder 精排。

In [ ]:
recall_method_records, recall_per_query, recall_summary, recall_timing, recall_rankings = run_recall_layer_benchmark(eval_samples)

recall_summary = recall_summary.sort_values(['ndcg@10', 'mrr'], ascending=[False, False]).reset_index(drop=True)
recall_timing_summary = recall_timing.groupby('method')['latency_seconds'].agg(['mean', 'median', 'max']).reset_index()

display(recall_summary[['method', 'precision@5', 'recall@5', 'hit@5', 'ndcg@5', 'precision@10', 'recall@10', 'hit@10', 'ndcg@10', 'mrr']])
display(recall_timing_summary)


## Controlled End-to-End Comparison

这一节只换召回器，后面的 `Cross-Encoder + business score` 保持不变。

In [ ]:
end_method_records, end_per_query, end_summary, end_timing, end_rankings = run_end_to_end_benchmark(eval_samples)

end_summary = end_summary.sort_values(['ndcg@10', 'mrr'], ascending=[False, False]).reset_index(drop=True)
end_timing_summary = end_timing.groupby('method')['latency_seconds'].agg(['mean', 'median', 'max']).reset_index()
query_meta = build_query_metadata(eval_samples)

display(end_summary[['method', 'precision@5', 'recall@5', 'hit@5', 'ndcg@5', 'precision@10', 'recall@10', 'hit@10', 'ndcg@10', 'mrr']])
display(end_timing_summary)
print('NB fit time (seconds) =', round(nb_fit_seconds, 3))
print('Current pipeline init time (seconds) =', round(service_init_seconds, 3))


In [ ]:
current_per_query = end_per_query[end_per_query['method'] == 'current_full_pipeline'].copy()
nb_per_query = end_per_query[end_per_query['method'] == 'nb_text_labels_same_rerank'].copy()

delta_df = current_per_query.merge(
    nb_per_query,
    on='query_id',
    suffixes=('_current', '_nb'),
).merge(query_meta, on='query_id', how='left')

delta_df['ndcg@10_delta'] = delta_df['ndcg@10_current'] - delta_df['ndcg@10_nb']
delta_df['mrr_delta'] = delta_df['mrr_current'] - delta_df['mrr_nb']
delta_df['hit@10_delta'] = delta_df['hit@10_current'] - delta_df['hit@10_nb']

display(delta_df[['query_id', 'ndcg@10_current', 'ndcg@10_nb', 'ndcg@10_delta', 'mrr_current', 'mrr_nb', 'mrr_delta', 'symptom_text', 'relevant_drugs']].sort_values('ndcg@10_delta', ascending=False).head(10))


In [ ]:
def pick_case_rows(delta_table: pd.DataFrame) -> list[tuple[str, pd.Series]]:
    picks: list[tuple[str, pd.Series]] = []
    used_query_ids: set[str] = set()

    ordered = [
        ('Current pipeline wins most', delta_table.sort_values('ndcg@10_delta', ascending=False)),
        ('Closest tie', delta_table.assign(abs_delta=delta_table['ndcg@10_delta'].abs()).sort_values('abs_delta', ascending=True)),
        ('NB wins most', delta_table.sort_values('ndcg@10_delta', ascending=True)),
    ]

    for label, table in ordered:
        for _, row in table.iterrows():
            if row['query_id'] not in used_query_ids:
                picks.append((label, row))
                used_query_ids.add(row['query_id'])
                break
    return picks


for title, row in pick_case_rows(delta_df):
    query_id = row['query_id']
    print(f"\n### {title}: {query_id}")
    print('symptom_text =', row['symptom_text'])
    print('relevant_drugs =', row['relevant_drugs'])
    print('ndcg@10 delta =', round(row['ndcg@10_delta'], 4), '| mrr delta =', round(row['mrr_delta'], 4))
    print('\nCurrent pipeline top-5')
    display(top_drugs(end_rankings, query_id, 'current_full_pipeline', top_n=5))
    print('NB + same rerank top-5')
    display(top_drugs(end_rankings, query_id, 'nb_text_labels_same_rerank', top_n=5))


## Interpretation Guide

- 如果 `NB-text+labels` 在召回层明显落后于 `dual recall without rerank`，说明当前双召回器本身就在改善候选集质量。
- 如果 `NB-text+labels -> same rerank` 仍明显落后于 `current_full_pipeline`，说明差距主要来自召回阶段，而不是后续精排。
- 如果两者差距缩小，说明当前系统的主要收益更可能来自 `Cross-Encoder`，而不是召回器本身。
- `NB-text` 不作为 headline result，它只是帮助说明“只靠自然语言文本 + 传统概率模型”时，效果大概处在哪个级别。